Мат модель для нахождения коэффициента влияния гендера на ЗП

In [ ]:
import pandas as pd

modelv10 = pd.read_excel("../../data/processed/v10_clean_base.xlsx")

modelv10 v1.0 (линейная регрессия по параметрам: salary, gender, experience, education_type, region_code, industry_code)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

modelv10 = modelv10[
    (modelv10['salary'] > 5000) &
    (modelv10['salary'] < 200000) &
    (modelv10['experience_mistake'] != 1) &
    (modelv10['birthday_mistake'] != 1) &
    (modelv10['gender'].isin(['Мужской', 'Женский'])) &
    (modelv10['industry_code'] == 'BuldindRealty') &
    (modelv10['region_code'].notna())
].copy()

modelv10['male'] = (modelv10['gender'] == 'Мужской').astype(int)
modelv10['experience'] = modelv10['experience'].fillna(0)

results = []

for code, grp in modelv10.groupby('region_code'):
    nm = (grp['gender'] == 'Мужской').sum()
    nf = (grp['gender'] == 'Женский').sum()
    if nm < 5 or nf < 5:
        continue

    modelv10 = smf.ols('salary ~ male + experience', data=grp).fit()

    results.append({
        'регион': int(code),
        'n_М': nm,
        'n_Ж': nf,
        'n_всего': len(grp),
        'коэф_gender': round(modelv10.params['male'], 0),
        'p_gender': round(modelv10.pvalues['male'], 4),
        'gender_значимо': 'да' if modelv10.pvalues['male'] < 0.05 else 'нет',
        'коэф_опыт': round(modelv10.params['experience'], 0),
        'p_опыт': round(modelv10.pvalues['experience'], 4),
        'R2': round(modelv10.rsquared, 3)
    })

out = pd.DataFrame(results).sort_values('p_gender')
print(out.to_string(index=False))
out.to_excel('all_regions_regression.xlsx', index=False)